### 05C · Modelo de ruido sin mapeo: TAR autoexcitado (régimen de f × signo del residuo)
Variante del 05 (Hs). r = Z − f, con f fuera de muestra (OOF 1985–2015, modelo final 2016–2025) del 04.
- **Motivo (27/11):** con 2 regímenes (calma/temporal) las innovaciones tienen asimetría +1,4 y el residuo −0,33, así que sin corrección marginal el TAR da asimetría positiva (temporales +24 %) y el 05 necesita el mapeo por rangos. Partiendo por el signo de r(t−1): con r(t−1) < 0 (f se pasa) las innovaciones tienen σ 0,081 y asimetría +2,2 (el error se cierra a saltos: desfases de f); con r(t−1) > 0, σ 0,061 y asimetría −0,3.
- **Dinámica:** AR(24) distinto en 4 regímenes: (calma f < q90 / temporal f ≥ q90) × (r(t−1) < 0 / > 0). Retardos de la serie entera; innovaciones sueltas remuestreadas por régimen.
- **Sin mapeo ni transformación:** r = salida directa del TAR. Prueba rápida (1 sim sobre la f del OOF): asimetría −0,12 (real −0,33; con 2 regímenes +0,31), forma por decil correcta, σ 0,541 = real, ACF 12/24 h igual.
- **Hindcast:** f histórica 1985–2025 + 50 realizaciones → Hs, frente al observado.

Salidas: `outputs/ruido/ruido_tar_setar_{point}.pkl` y `outputs/ruido/hs_hindcast_setar_{point}.pkl`. No toca las del 05 (producción).

In [ ]:
import os, time, pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.stats import genextreme, skew

from utils.tv_var.periodic_var import PeriodicVAR, select_order
from utils.pit.piece_wise_marginal import from_normal

#### Datos

In [ ]:
point = 'p102'
HINDCAST = ('1985', '2025')                    # periodo del hindcast completo
TEST = ('2016', '2025')                        # periodo fuera de muestra para el XGB y para el ruido

resid = pd.read_csv(f"outputs/xgb_{point}/residuos_horarios.csv", index_col=0, parse_dates=True)
hs_obs = pd.read_csv(f"inputs/waves_bulk_{point}.csv", index_col=0, parse_dates=True)["VHM0"]

tr_ = pickle.load(open(f"outputs/marginal/transformacion_{point}.pkl", "rb"))
marg, harm_z = tr_["marg"], tr_["harm_z"]

def hs_desde_z(z):
    """Serie de Z (Hs) → Hs en metros, deshaciendo la armónica, Φ y la marginal."""
    zz = pd.DataFrame({"VHM0": z, "VTM02": 0.0}, index=z.index)      # la armónica se ajustó con las dos columnas
    z0 = harm_z.inverse_transform(zz)["VHM0"].clip(-6.36, 6.36)    # mismo límite que to_normal (eps=1e-10)
    return pd.Series(marg["VHM0"].ppf(from_normal(z0).values), index=z.index)

print(f"std r oof {resid.loc[resid.periodo == 'oof', 'r'].std():.3f} | test {resid.loc[resid.periodo == 'test', 'r'].std():.3f}")

#### Residuo de ajuste (OOF 1985–2015) y deciles de f

In [ ]:
r_oof = resid.loc[resid.periodo == "oof", ["r"]].asfreq("h")          # malla horaria completa (huecos = NaN)
f_oof = resid["f"].reindex(r_oof.index)

bordes = np.quantile(f_oof.dropna(), [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])   # 9 bordes → 10 deciles (se reutilizan al simular)
dec_oof = pd.Series(np.digitize(f_oof, bordes), index=r_oof.index).where(f_oof.notna())   # 0 … 9

#### Orden del AR
El BIC prefiere 48, pero de 24 a 48 la verosimilitud mejora un 0,08 % y los temporales no cambian (registro 25/11: AR(48)/AR(24)/AR(12) iguales). Se usa 24.

In [ ]:
select_order(r_oof, p_values=[12, 20, 24, 36, 48], n_harm=0, freq="h")

#### TAR autoexcitado: AR(24) en 4 regímenes, (calma / temporal) × (r(t−1) < 0 / > 0)
Una sola matriz de retardos con la serie entera (`_lag_matrix`, retardos por tiempo); cada régimen se ajusta con las filas cuya hora t cae en él. El signo es el del residuo de la hora anterior (`r_lag_1`); al simular, el del propio ruido simulado.

In [ ]:
P = 24
U_REG = f_oof.quantile(0.90)
NOMBRES = {0: "calma, r(t−1) < 0", 1: "calma, r(t−1) > 0", 2: "temporal, r(t−1) < 0", 3: "temporal, r(t−1) > 0"}

ar_unico = PeriodicVAR(p=P, n_harm=0, freq="h").fit(r_oof)            # AR único: referencia y da self.dt
X_lag, Y = ar_unico._lag_matrix(r_oof, P)                              # const + r_lag_1 … r_lag_24, por tiempo
reg_fila = (2 * (f_oof.reindex(X_lag.index) >= U_REG).astype(int)
            + (X_lag["r_lag_1"] > 0).astype(int))                       # 0 … 3

coef, innov = {}, {}
for j, nombre in NOMBRES.items():
    m = (reg_fila == j).values
    coef[j] = np.linalg.lstsq(X_lag.values[m], Y.values[m, 0], rcond=None)[0]
    innov[j] = Y.values[m, 0] - X_lag.values[m] @ coef[j]
    print(f"{nombre:22s} n {m.sum():6d} | σ innov {innov[j].std():.3f} | asimetría {skew(innov[j]):+.2f} | "
          f"suma coefs AR {coef[j][1:].sum():.3f} | const {coef[j][0]:+.4f}")

In [ ]:
#### Innovaciones del TAR: escala por decil de f y agrupamiento de volatilidad
e_tar = pd.Series(np.nan, index=X_lag.index)
for j in NOMBRES:
    e_tar[(reg_fila == j).values] = innov[j]
e_tar = e_tar.asfreq("h")                                              # malla horaria: el lag 1 es 1 h de verdad

sd_dec = e_tar.groupby(dec_oof.reindex(e_tar.index)).std()
print(sd_dec.round(3).values)
print(f"ratio decil 10 / decil 1: {sd_dec.iloc[-1] / sd_dec.iloc[0]:.2f}")
print(f"ACF de e² en lag 1: todo {(e_tar**2).autocorr(1):.3f} | solo DEF {(e_tar[e_tar.index.month.isin([12, 1, 2])]**2).autocorr(1):.3f}")
#### Hay agrupamiento de volatilidad real (no es estacional). Se deja con innovaciones sueltas (L = 1);
#### si el hindcast muestra sesgos en duraciones o forma de los picos, probar bloques de 6 h por régimen.

#### Ruido TAR autoexcitado → Hs (sin mapeo)

In [ ]:
def ruido_tar(f, rng, burn=24 * 30):
    """TAR hora a hora: régimen = (calma/temporal según f) × (signo del ruido simulado en t−1), innovaciones sueltas de su régimen."""
    gf = (f.values >= U_REG).astype(int)
    gf = np.r_[np.full(burn, gf[0]), gf]                                # arranque con el régimen de f de la primera hora
    c0 = np.array([coef[k][0] for k in NOMBRES])
    a = np.stack([coef[k][1:] for k in NOMBRES])
    e = np.stack([innov[k][rng.integers(len(innov[k]), size=len(gf))] for k in NOMBRES])   # (4, T): un sorteo por régimen y hora
    x = np.zeros(len(gf) + P)
    for t in range(len(gf)):
        k = 2 * gf[t] + (x[t + P - 1] > 0)
        x[t + P] = c0[k] + a[k] @ x[t:t + P][::-1] + e[k, t]
    return x[P + burn:]

def hs_desde_f(f, rng):
    r = pd.Series(ruido_tar(f, rng), index=f.index)                    # sin mapeo: el ruido es la salida del TAR
    return hs_desde_z(f + r), r

#### Hindcast: f histórica (OOF + test) + N realizaciones de ruido

In [ ]:
f_sim = resid.loc[HINDCAST[0]:HINDCAST[1], "f"].asfreq("h").interpolate()   # malla horaria continua
hs_f = hs_desde_z(f_sim)                                                     # f sola, en metros

n_sims = 50
hs_sims, r_sims = [], []
for i in range(n_sims):
    t = time.time()
    h, r = hs_desde_f(f_sim, np.random.default_rng(i))
    hs_sims.append(h.rename(i))
    r_sims.append(r.rename(i))
    print(f"sim {i:02d} ({time.time() - t:.0f} s)")

hs_sims = pd.concat(hs_sims, axis=1)                                         # (horas, n_sims)
r_sims = pd.concat(r_sims, axis=1)

#### Validación: observado vs simulado (mismas horas que el observado)

In [ ]:
hs_obs_h = hs_obs.reindex(f_sim.index)                                       # mismo periodo y malla que la simulación
hay_obs = hs_obs_h.notna()
U90, U99 = hs_obs_h.quantile(0.90), hs_obs_h.quantile(0.99)
print(f"U90 {U90:.2f} m | U99 {U99:.2f} m")

def est4(ix):
    return np.select([ix.month.isin([12, 1, 2]), ix.month.isin([3, 4, 5]), ix.month.isin([6, 7, 8])],
                     ["invierno", "primavera", "verano"], "otoño")

def picos_temporal(s):
    """Picos de temporal: Hs > U99, separados 72 h (definición oficial)."""
    return s.index[find_peaks(s.fillna(0).values, height=U99, distance=72)[0]]

def duraciones(s, u):
    d = np.diff(np.r_[0, (s.values > u).astype(int), 0])
    return np.flatnonzero(d == -1) - np.flatnonzero(d == 1)

def metricas(s, gev=True):
    n_dec = s.dropna().index.year.nunique() / 10
    pk = picos_temporal(s)
    out = {"q99": s.quantile(0.99), "q99.9": s.quantile(0.999), "temporales / década": len(pk) / n_dec,
           **{f"  {e}": (est4(pk) == e).sum() / n_dec for e in ["invierno", "primavera", "verano", "otoño"]},
           "duración > U99 [h]": duraciones(s, U99).mean(), "duración > U90 [h]": duraciones(s, U90).mean()}
    if gev:
        c, loc, sc = genextreme.fit(s.groupby(s.index.year).max().dropna().values)
        out.update({f"Hs{T} GEV": genextreme.isf(1 / T, c, loc, sc) for T in [2, 10, 50]})
    return out

def tabla(a, b, gev=True):
    obs = pd.Series(metricas(hs_obs_h.loc[a:b], gev))
    sim = pd.DataFrame([metricas(hs_sims[c].where(hay_obs).loc[a:b], gev) for c in hs_sims.columns])
    return pd.DataFrame({"observado": obs, "p5": sim.quantile(0.05), "p50": sim.median(), "p95": sim.quantile(0.95),
                         "p50 / obs": sim.median() / obs})

display(tabla(*HINDCAST).round(2))
tabla(*TEST, gev=False).round(2)

In [ ]:
#### Temporales por mes: observado, f sola y mediana simulada
def temporales_mes(s):
    pk = picos_temporal(s)
    return pd.Series(pk.month).value_counts().reindex(range(1, 13), fill_value=0) * 10 / s.dropna().index.year.nunique()

pd.DataFrame({"observado": temporales_mes(hs_obs_h),
              "f sola": temporales_mes(hs_f.where(hay_obs)),
              "sim p50": pd.concat([temporales_mes(hs_sims[c].where(hay_obs)) for c in hs_sims.columns], axis=1).median(axis=1)}
            ).T.round(1)

In [ ]:
#### Cobertura de la banda 95 % (esperado 2,5 % por encima y 2,5 % por debajo)
q_lo, q_hi = hs_sims.quantile(0.025, axis=1), hs_sims.quantile(0.975, axis=1)
for a, b in [HINDCAST, TEST]:
    o = hs_obs_h.loc[a:b].dropna()
    print(f"{a}–{b}: por encima {(o > q_hi.loc[o.index]).mean():.1%} | por debajo {(o < q_lo.loc[o.index]).mean():.1%}")

#### Guardar el modelo de ruido (prueba: no pisa la producción del 05)

In [ ]:
os.makedirs("outputs/ruido", exist_ok=True)
pickle.dump({"P": P, "U_REG": U_REG, "coef": coef, "innov": innov, "regimenes": NOMBRES, "bordes": bordes},
            open(f"outputs/ruido/ruido_tar_setar_{point}.pkl", "wb"))
hs_sims.astype("float32").to_pickle(f"outputs/ruido/hs_hindcast_setar_{point}.pkl")

#### Figuras

In [ ]:
os.makedirs("figuras", exist_ok=True)
TINTA, AZUL, NARANJA, GRIS, FONDO = "#13233A", "#2C6E91", "#C8622A", "#7A8594", "#F5F3EE"
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Inter", "Arial", "DejaVu Sans"],
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA, "text.color": TINTA, "xtick.color": GRIS, "ytick.color": GRIS,
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.spines.top": False, "axes.spines.right": False,
})

def acf_h(s, lags):
    x = s.asfreq("h").values.astype("float64"); x = x - np.nanmean(x); v = np.nanmean(x * x)
    return np.array([1.0 if l == 0 else np.nanmean(x[:-l] * x[l:]) / v for l in lags])

deciles = np.arange(1, 11)

In [ ]:
#### Residuo por decil de f (OOF): la forma que tiene que reproducir el TAR
d_oof = resid.loc[resid.periodo == "oof", ["f", "r"]].dropna()
d_oof["decil"] = np.digitize(d_oof["f"], bordes) + 1
datos = [d_oof.loc[d_oof.decil == k, "r"].values for k in deciles]

fig, ax = plt.subplots(figsize=(5.6, 2.4), constrained_layout=True)
bp = ax.boxplot(datos, widths=0.6, showfliers=False, patch_artist=True, whis=(1, 99),
                medianprops=dict(color=TINTA, lw=1.5), whiskerprops=dict(color=GRIS),
                capprops=dict(color=GRIS), boxprops=dict(edgecolor=TINTA, lw=0.8))
for caja in bp["boxes"]:
    caja.set_facecolor(AZUL); caja.set_alpha(0.8)
ax.axhline(0, color=NARANJA, lw=1.2, ls="--", zorder=0)
ytop = ax.get_ylim()[1]
for k, v in zip(deciles, datos):
    ax.text(k, ytop, f"{v.std():.2f}", ha="center", va="bottom", fontsize=8, color=GRIS)
ax.set(xlabel="decil de ŷ", ylabel="r = Z − ŷ", xticks=deciles)
fig.savefig("figuras/05C_ruido_residuo_deciles.png", dpi=200)
plt.show()

In [ ]:
#### Diagnóstico (26/11): dentro del decil 10, ¿depende r de f? → la media cae de +0.07 a −0.02 en el 2 % más alto,
#### pero clases más finas arriba (q95, q98) no arreglaron el nº de temporales y acortaron las duraciones (0.90): se quedan 10 deciles
top = d_oof[d_oof.decil == 10].copy()
top["sub"] = pd.qcut(top["f"], 5, labels=["q90–92", "q92–94", "q94–96", "q96–98", "q98–100"])
print(top.groupby("sub")["r"].agg(["mean", "std", "count"]).round(3))

In [ ]:
#### σ y asimetría del residuo por decil de f: real (test) vs simulado
d = pd.DataFrame({"f": f_sim, "r_real": resid["r"].reindex(f_sim.index)}).loc[TEST[0]:TEST[1]]
d["decil"] = np.digitize(d["f"], bordes) + 1
g = r_sims.loc[d.index].groupby(d["decil"])

sd_real, sk_real = d.groupby("decil")["r_real"].std(), d.groupby("decil")["r_real"].apply(lambda x: skew(x.dropna()))
sd_sim, sk_sim = g.std(), g.agg(lambda x: skew(x))                     # (decil, sim)

fig, axs = plt.subplots(1, 2, figsize=(5.6, 2.4), constrained_layout=True, sharex=True)
for ax, real, sim, lab in [(axs[0], sd_real, sd_sim, "σ del residuo"), (axs[1], sk_real, sk_sim, "asimetría")]:
    p5, p50, p95 = np.percentile(sim.values, [5, 50, 95], axis=1)
    ax.fill_between(sim.index, p5, p95, color=AZUL, alpha=0.25, lw=0)
    ax.plot(sim.index, p50, color=AZUL, lw=1.6, label="simulado")
    ax.plot(real.index, real.values, "o-", color=TINTA, lw=1.4, ms=3.5, label="real (test)")
    ax.set(title=lab, xlabel="decil de ŷ", xticks=[1, 4, 7, 10])
axs[1].axhline(0, color="#DDD8CE", lw=0.8, zorder=0)
axs[0].legend(frameon=False, fontsize=8)
fig.savefig("figuras/05C_ruido_sigma_asimetria.png", dpi=200)
plt.show()

In [ ]:
#### Persistencia del residuo: real (test) vs TAR simulado
lags = np.arange(0, 121)                                # 5 días
acf_real = acf_h(resid.loc[TEST[0]:TEST[1], "r"], lags)
acf_sim = np.array([acf_h(r_sims[c].loc[TEST[0]:TEST[1]], lags) for c in r_sims.columns])
p5, p95 = np.percentile(acf_sim, [5, 95], axis=0)

fig, ax = plt.subplots(figsize=(5.6, 2.4), constrained_layout=True)
ax.fill_between(lags / 24, p5, p95, color=AZUL, alpha=0.3, lw=0, label="TAR autoexcitado simulado p5–p95")
ax.plot(lags / 24, acf_real, color=TINTA, lw=1.8, label="residuo real (test)")
ax.axhline(0, color="#DDD8CE", lw=0.8)
ax.set(xlabel="desfase (días)", ylabel="autocorrelación")
ax.legend(frameon=False, fontsize=8)
fig.savefig("figuras/05C_ruido_acf.png", dpi=200)
plt.show()

In [ ]:
#### Hindcast en dos temporales del test: Hs observada vs banda de las realizaciones
ventanas = [("2020-01-05", "2020-02-05", ("Gloria", "2020-01-21")),
            ("2023-02-09", "2023-03-20", ("Juliette", "2023-02-27"))]

fig, axs = plt.subplots(len(ventanas), 1, figsize=(11.6, 2.6 * len(ventanas)), constrained_layout=True)
for ax, (t0, t1, (nombre, fecha)) in zip(axs, ventanas):
    w = hs_sims.loc[t0:t1]
    ax.fill_between(w.index, w.quantile(0.025, axis=1), w.quantile(0.975, axis=1), color=AZUL, alpha=0.25, lw=0,
                    label="ŷ + ruido, banda 95 %")
    ax.plot(w.iloc[:, 0], color=AZUL, lw=0.8, alpha=0.8, label="una realización")
    ax.plot(hs_f.loc[t0:t1], color=NARANJA, lw=1.6, ls="--", label="ŷ en metros")
    ax.plot(hs_obs_h.loc[t0:t1], color=TINTA, lw=1.3, label="observado")
    ax.axhline(U99, color=GRIS, lw=0.8, ls=":")
    ax.text(pd.Timestamp(fecha), ax.get_ylim()[1], f" {nombre}", color=NARANJA, fontsize=10, va="top")
    ax.set(ylabel="Hs (m)", xlim=(pd.Timestamp(t0), pd.Timestamp(t1)))
    ax.grid(axis="y", color="#DDD8CE", lw=0.8); ax.set_axisbelow(True)
axs[0].legend(frameon=False, loc="upper right", ncol=4, fontsize=9)
fig.savefig("figuras/05C_ruido_temporales_hindcast.png", dpi=200)
plt.show()

In [ ]:
#### Nº de temporales y tamaño de los picos (hindcast completo)
p_obs = hs_obs_h.loc[picos_temporal(hs_obs_h)].values
p_sim = [hs_sims[c].loc[picos_temporal(hs_sims[c].where(hay_obs))].values for c in hs_sims.columns]
n_sim = [len(p) for p in p_sim]

fig, axs = plt.subplots(1, 2, figsize=(7.7, 2.0), constrained_layout=True, gridspec_kw={"width_ratios": [1.2, 1]})
axs[0].hist(n_sim, bins=15, color=AZUL, alpha=0.6, edgecolor=FONDO)
axs[0].axvline(len(p_obs), color=NARANJA, lw=2, label=f"observado ({len(p_obs)})")
axs[0].set(xlabel=f"nº de temporales (Hs > {U99:.2f} m)", ylabel="realizaciones")
axs[0].legend(frameon=False, fontsize=8)

for j, q in enumerate([0.5, 0.9]):
    qs = np.array([np.quantile(p, q) for p in p_sim])
    p5, p50, p95 = np.percentile(qs, [5, 50, 95])
    axs[1].errorbar(j, p50, yerr=[[p50 - p5], [p95 - p50]], fmt="o", color=AZUL, capsize=4, lw=1.5,
                    label="simulado p5–p95" if j == 0 else None)
    axs[1].plot(j, np.quantile(p_obs, q), "D", color=NARANJA, ms=7, label="observado" if j == 0 else None)
axs[1].set_xticks([0, 1], ["pico q50", "pico q90"])
axs[1].set(ylabel="Hs (m)", xlim=(-0.6, 1.6))
axs[1].legend(frameon=False, fontsize=8)
fig.savefig("figuras/05C_ruido_temporales_n.png", dpi=200)
plt.show()

In [ ]:
#### Estacionalidad del residuo: reparto de la varianza de Z y σ(r) por mes
meses = np.arange(1, 13)
etq = ["E", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]

oof = resid.loc[resid.periodo == "oof", ["z", "f", "r"]].dropna()
g = oof.groupby(oof.index.month)
var_f, var_r = g["f"].var(), g["r"].var()
r2 = g.apply(lambda x: x["z"].corr(x["f"]) ** 2)

sd_oof = g["r"].std()
sd_test = resid.loc[resid.periodo == "test", "r"].groupby(lambda t: t.month).std()
r_test = r_sims.loc[TEST[0]:TEST[1]]
sd_sim = r_test.groupby(r_test.index.month).std()                    # (mes, realización)
p5, p50, p95 = np.percentile(sd_sim.values, [5, 50, 95], axis=1)

fig, axs = plt.subplots(1, 2, figsize=(11.6, 3.4), constrained_layout=True)
axs[0].bar(meses, var_f, color=AZUL, width=0.7, label="var(ŷ)  explicada por el XGBoost")
axs[0].bar(meses, var_r, bottom=var_f, color=NARANJA, alpha=0.8, width=0.7, label="var(r)  residuo")
axs[0].axhline(1, color=TINTA, lw=1, ls="--")
for m in meses:
    axs[0].text(m, var_f[m] / 2, f"{r2[m]:.2f}", ha="center", va="center", fontsize=8, color="white")
axs[0].set(xticks=meses, xticklabels=etq, ylim=(0, 1.15), ylabel="varianza de Z",
           title="Reparto de la varianza (R² dentro de las barras)")
axs[0].legend(frameon=False, fontsize=8, loc="upper center", ncol=2, bbox_to_anchor=(0.5, -0.12))

axs[1].fill_between(meses, p5, p95, color=AZUL, alpha=0.25, lw=0, label="simulado p5–p95 (test)")
axs[1].plot(meses, p50, color=AZUL, lw=1.8, label="simulado mediana (test)")
axs[1].plot(meses, sd_oof.reindex(meses), "o-", color=TINTA, lw=1.5, ms=4, label="real 1985–2015 (OOF)")
axs[1].plot(meses, sd_test.reindex(meses), "s--", color=NARANJA, lw=1.3, ms=4, label="real test 2016–2025")
axs[1].set(xticks=meses, xticklabels=etq, ylabel="σ del residuo (Z)", title="σ del residuo por mes")
axs[1].legend(frameon=False, fontsize=8, loc="upper center", ncol=2, bbox_to_anchor=(0.5, -0.12))
for ax in axs:
    ax.grid(axis="y", color="#DDD8CE", lw=0.8); ax.set_axisbelow(True)
fig.savefig("figuras/05C_ruido_estacionalidad.png", dpi=200)
plt.show()